### Plot grit scores for Spheroid Aggregated data

In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, derived, require)
from utils.panels import save_panel

import pandas as pd
import numpy as np
import os

# Grit scores
from cytominer_eval import evaluate

# Plotting
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns; sns.set_style("white")


# Set current working directory


In [ ]:
# Set up the plotting parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
dpi = 300
figformat = 'pdf'

In [ ]:
# Save the data
ImagesOut = str(figdir('Fig5')) + '/'

if not os.path.exists(ImagesOut): 
        os.makedirs(ImagesOut)

In [ ]:
# Parameters. run_all.py sweeps these via the environment so one notebook can
# emit every (cell_line, data_type) combination; the defaults keep interactive
# use unchanged.
import os
cell_line = os.environ.get('COLOPAINT3D_CELL_LINE', 'HCT116')   # 'HCT116' or 'HT29'
data_type = os.environ.get('COLOPAINT3D_DATA_TYPE', 'MIP')      # 'MIP', 'aggregates' or '2D'
print(f'cell_line={cell_line}  data_type={data_type}')


In [ ]:
# # Load the data
dir = str(profiles("exp1_main", "")) + "/"
data =  pd.read_parquet(('{}selected_data_{}_{}.parquet').format(dir, data_type, cell_line))

In [ ]:
# Some function definitions

def list_features(df):
    # List features
    list_of_selected_features = list(df.columns.values)
    list_of_metadata = list(df.columns[df.columns.str.contains("Metadata_")])
    list_of_selected_features = list(set(list_of_selected_features) - set(list_of_metadata))
    
    return list_of_selected_features, list_of_metadata

### Grit

In [ ]:
## Prepare the metadata for the grit calculation
dataset_grit = data.copy()

dataset_grit['Metadata_name'] = dataset_grit['Metadata_cmpdname'].str[:5] 

# Add a column with the step of the concentration (easier for plotting)
dataset_grit['Metadata_conc_step'] = (
    dataset_grit.groupby('Metadata_cmpdname')['Metadata_cmpd_conc'].rank(ascending=True, method='dense')
    )
# Add a column with the name of the perturbation (compound + concentration)
dataset_grit["Metadata_pert_name"] = (
    dataset_grit["Metadata_name"] + "_" + dataset_grit["Metadata_cmpd_conc"].astype(str)
    )
# Add a column with the a unique identifier for each replicate
dataset_grit["Metadata_replicate_id"] = (
    dataset_grit["Metadata_name"] + "_" + dataset_grit.index.astype(str)
    )

In [ ]:
# Setup the data for calculating grit

# Set up the input for evaluate
control_perts = dataset_grit.query("Metadata_name == 'dmso' & Metadata_cmpd_conc == 0.1").Metadata_replicate_id.unique().tolist()
grit_replicate_groups = {"profile_col": "Metadata_replicate_id", "replicate_group_col": "Metadata_pert_name"} 

In [ ]:
# Calcuate the grit scores

grit_scores = []

grit_results = evaluate(
    profiles=dataset_grit,
    features=list_features(dataset_grit)[0],
    meta_features=list_features(dataset_grit)[1],
    replicate_groups=grit_replicate_groups,
    operation="grit",
    similarity_metric="pearson",
    grit_replicate_summary_method="median",
    grit_control_perts=control_perts,
)

grit_scores.append(grit_results)

In [ ]:
# Organize the grit scores

grit_scores = pd.concat(grit_scores).reset_index(drop=True)
grit_scores["Metadata_name"] = grit_scores["perturbation"].str.split("_").str[0]

# Add back the well information
grit_scores = pd.merge(grit_scores,dataset_grit[['Metadata_PlateWell', 'Metadata_replicate_id','Metadata_cmpd_conc', 'Metadata_conc_step', 'Metadata_pert_type']], left_on='perturbation', right_on='Metadata_replicate_id')

# list the compounds
CompoundsUsed = grit_scores["Metadata_name"].unique()

In [ ]:
## Plot the grit dose response

# Plot the grit scores
fig = plt.figure(figsize=(48, 32))
sns.set(font_scale=1.5)
fig.suptitle("grit barplot ", fontsize=24, x=0.3)
sp = 1
nrrow = 6
nrcol = 11
for some in CompoundsUsed[:]:
    grit_scores_part = grit_scores[grit_scores["Metadata_name"] == some].copy()
    grit_scores_part.sort_values(by="Metadata_conc_step", inplace=True)
    ax = fig.add_subplot(nrrow, nrcol, sp)
    ax = sns.barplot(
        x="Metadata_cmpd_conc",
        y="grit",
        data=grit_scores_part,
        hue="Metadata_conc_step",
        legend=False,
        palette="Blues_d",
        alpha=1,
        err_kws={'linewidth': 3,'color': 'black'},
    )
    ax.set_facecolor("w")
    ax.spines["bottom"].set_color("grey")
    ax.spines["left"].set_color("grey")
    ax.set_ylim([-0.1, 7.5])
    ax.set_title("{}".format(some), fontsize=24, x=0.2)
    plt.subplots_adjust(top=0.9, wspace=0.2, hspace=0.5, left=0.0)
    plt.xticks(rotation = 45)
    sp += 1
    
# [not a paper panel] fig.savefig(
# [not a paper panel] "3_Figure3/GritScores/result-images/GritScores_{}_{}.{}".format(cell_line, data_type,'png'), dpi=dpi, bbox_inches="tight"
# [not a paper panel] )

plt.show()


In [ ]:
# # Plot only a handful of compounds for the main figure

# CompoundsUsed = ['SN-38', 'Binim', 'abema', 'etop','fenb','stau','dmso']


# # Plot the grit scores
# fig = plt.figure(figsize=(16, 4))
# # sns.set(font_scale=1.5)
# fig.suptitle("grit barplot ", fontsize=24, x=0.3)
# sp = 1
# nrrow = 1
# nrcol = 7
# for some in CompoundsUsed[:]:
#     grit_scores_part = grit_scores[grit_scores["Metadata_name"] == some].copy()
#     grit_scores_part.sort_values(by="Metadata_conc_step", inplace=True)
#     ax.axhline(y=1.96, color='black', linestyle='--', linewidth=1, alpha=0.7)
#     ax = fig.add_subplot(nrrow, nrcol, sp)
#     ax = sns.barplot(
#         x="Metadata_cmpd_conc",
#         y="grit",
#         data=grit_scores_part,
#         hue="Metadata_conc_step",
#         legend=False,
#         palette="Blues_d",
#         alpha=1,
#         err_kws={'linewidth': 2,'color': 'black'},
#     )
#     ax.set_facecolor("w")
#     ax.spines["bottom"].set_color("grey")
#     ax.spines["left"].set_color("grey")
#     ax.set_yticks([0, 2, 4, 6])
#     ax.set_ylim([-0.1, 7.5])
#     ax.set_xlabel('µM')
#     ax.set_title("{}".format(some), fontsize=20, x=0.2)
#     plt.subplots_adjust(top=0.9, wspace=0.2, hspace=0.5, left=0.0)
#     plt.xticks(rotation = 45)
#     sp += 1

#     fig.tight_layout()
    
# # fig.savefig(
# #         "{}GritScores_{}_{}.{}".format(ImagesOut, cell_line, data_type, figformat), dpi=dpi, bbox_inches="tight"
# #         )

# plt.show()

In [ ]:
# Plot only a handful of compounds for the main figure
CompoundsUsed = ['SN-38', 'Binim', 'abema', ['etop','fenb','stau','dmso']]

# Plot the grit scores
fig = plt.figure(figsize=(10, 4))
fig.suptitle("Grit Scores for {}, {}".format(cell_line, data_type), fontsize=24, x=0.3)
sp = 1
nrrow = 1
nrcol = 4

for item in CompoundsUsed:
    ax = fig.add_subplot(nrrow, nrcol, sp)
    
    if isinstance(item, list):  # Grouped compounds
        # Combine data for all compounds
        combined_data = []
        for compound in item:
            temp_data = grit_scores[grit_scores["Metadata_name"] == compound].copy()
            combined_data.append(temp_data)
        
        combined_data = pd.concat(combined_data)
        combined_data = combined_data.sort_values(['Metadata_name', 'Metadata_cmpd_conc'])
        
        # Create the barplot with compound names as hue
        n_compounds = len(item)
        colors = plt.cm.Blues(np.linspace(0.3, 0.9, n_compounds))
        color_dict = dict(zip(item, colors))
        
        sns.barplot(
            x="Metadata_name",
            y="grit",
            hue="Metadata_name",
            data=combined_data,
            palette=color_dict,
            alpha=1,
            err_kws={'linewidth': 1.5,'color': 'black'},
            ax=ax,
            order=item,
            hue_order=item,
            legend=False
        )
        
        ax.set_title("controls", fontsize=16)
        ax.set_xlabel('Compound')
        
    else:  # Single compound
        grit_scores_part = grit_scores[grit_scores["Metadata_name"] == item].copy()
        grit_scores_part.sort_values(by="Metadata_conc_step", inplace=True)
        
        sns.barplot(
            x="Metadata_cmpd_conc",
            y="grit",
            data=grit_scores_part,
            hue="Metadata_conc_step",
            legend=False,
            palette="Blues_d",
            alpha=1,
            err_kws={'linewidth': 2,'color': 'black'},
            ax=ax
        )
        ax.set_title("{}".format(item), fontsize=20, x=0.2)
        ax.set_xlabel('µM')
    
    # Common formatting
    ax.axhline(y=1.96, color='black', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_facecolor("w")
    ax.spines["bottom"].set_color("grey")
    ax.spines["left"].set_color("grey")
    ax.set_yticks([0, 2, 4, 6])
    ax.set_ylim([-0.1, 7.5])
    plt.xticks(rotation=45)
    
    sp += 1

plt.subplots_adjust(top=0.9, wspace=0.3, hspace=0.5, left=0.0)
fig.tight_layout()

# [not a paper panel] fig.savefig(
# [not a paper panel] "{}GritScores_{}_{}.{}".format(ImagesOut, cell_line, data_type, figformat), dpi=dpi, bbox_inches="tight"
# [not a paper panel] )

plt.show()

#### Now add grit scores back to the data

In [ ]:
# Add the grit scores to selected_df
dataset_out = dataset_grit.merge(grit_scores[['Metadata_PlateWell', 'Metadata_replicate_id', 'grit']], left_on='Metadata_PlateWell', right_on = 'Metadata_PlateWell')

# Rename the grit column to Metadata_grit
dataset_out = dataset_out.rename(columns={'grit':'Metadata_grit'})

# Save the data
OutputDir = str(derived("exp1_main")) + "/"   # derived/, never the deposit

# Save as parquet
dataset_out.to_parquet(('{}grit_data_{}_{}.parquet').format(OutputDir, data_type, cell_line))


#### 2D vs aggregates compound overlap — moved to Figure 5a

This notebook used to end with a cell that rebuilt the 2D-vs-aggregates compound
overlap (published as **Fig 5a**) from four
`grit_scores_descriptive_stats_{data_type}_{cell_line}.csv` files. It has been removed:

* **The cell that wrote those CSVs does not survive in any source tree**, so the four
  inputs could never be produced here — every sweep combination read all four and
  failed on the first one missing.
* The analysis is not lost. `analysis/3_Figure5/3_Fig5a_grit_overlap.ipynb` derives the
  same compound sets from the deposited `grit_data_{data_type}_{cell_line}.parquet`
  tables — treatments whose median grit per perturbation exceeds 1.96 — and that
  definition was validated against all four surviving upstream CSVs, reproducing every
  set exactly (HCT116 2D 46/46, HT29 2D 47/47, HCT116 aggregates 38/38,
  HT29 aggregates 33/33).

What this notebook still does is write `grit_data_{data_type}_{cell_line}.parquet` to
`derived/`, which is what Fig 5a and Fig 3c/3d consume.
